# 01 -- Baseline Fine-Tuning: EuroBERT-210m

Single-label classification (13 classes) using HuggingFace Trainer API.
Fine-tuning on labeled German news articles (2025 Bundestagswahl) with default hyperparameters.

**Pipeline:**
1. 3-Fold Stratified Cross-Validation to estimate baseline performance (mean ± std)
2. Final retrain on all non-test data with NaN retry logic
3. Test set evaluation
4. Push trained model to HuggingFace Hub

**Prerequisites:** GPU runtime enabled (T4 / L4), `HF_TOKEN` in Colab Secrets.

In [ ]:
# === SETUP (identical in every notebook) ===
import os, sys

# Clone / update repo
REPO = "/content/news_articles_classification_thesis"
if not os.path.exists(REPO):
    !git clone https://github.com/ZorbeyOezcan/news_articles_classification_thesis.git {REPO}
else:
    !cd {REPO} && git pull -q

# Dependencies
!pip install -q transformers[sentencepiece] datasets huggingface_hub scikit-learn matplotlib seaborn tqdm pandas accelerate evaluate

# Mount Google Drive (persistent reports)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# Make pipeline_utils and hpt_utils importable
PIPELINE_DIR = f"{REPO}/Python/classification_pipeline/euroBERT_210m"
if PIPELINE_DIR not in sys.path:
    sys.path.insert(0, PIPELINE_DIR)

import importlib
import pipeline_utils as pu
import hpt_utils as hu
importlib.reload(pu)
importlib.reload(hu)

# Create output directories on Google Drive
pu.ensure_drive_dirs()

# HuggingFace Login
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get("HF_TOKEN"))

print(f"Reports directory: {pu.REPORTS_DIR}")
print("Setup complete.")

In [ ]:
# ===== MODEL & TRAINING CONFIGURATION =====
import torch
import numpy as np

MODEL_ID = "EuroBERT/EuroBERT-210m"
MODEL_SHORT_NAME = "eurobert_210m"
MODEL_TYPE = "fine-tuned"
REPO_NAME = "Zorryy/NewsBERT-germ-210m_baseline"

# ----- Baseline hyperparameters (no HPT -- default values) -----
MAX_LENGTH = 2048
NUM_EPOCHS = 8
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.06
WEIGHT_DECAY = 0.01
LABEL_SMOOTHING = 0.0
LR_SCHEDULER_TYPE = "linear"
MAX_GRAD_NORM = 1.0

# ----- Cross-validation -----
N_FOLDS = 3

# ----- Split configuration -----
TEST_PER_CLASS = 30
VAL_FRACTION = 0.2
RANDOM_SEED = 42

# ----- Fixed training settings -----
EFFECTIVE_BATCH_SIZE = 16
EARLY_STOPPING_PATIENCE = 3
LOGGING_STEPS = 50
MAX_NAN_RETRIES = 3

# ----- GPU-adaptive settings -----
if not torch.cuda.is_available():
    raise RuntimeError("GPU required! Please change Colab runtime type.")

_gpu_cap = torch.cuda.get_device_capability()
_gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9

# Mixed precision: BF16 for Ampere+ (L4, A100), FP16 for T4
if _gpu_cap[0] >= 8:
    USE_BF16 = True
    USE_FP16 = False
else:
    USE_BF16 = False
    USE_FP16 = True

# Batch sizes per GPU tier
if _gpu_mem >= 40:       # A100
    BATCH_SIZE_TRAIN = 8
    BATCH_SIZE_EVAL = 32
elif _gpu_mem >= 20:     # L4
    BATCH_SIZE_TRAIN = 4
    BATCH_SIZE_EVAL = 16
else:                    # T4
    BATCH_SIZE_TRAIN = 2
    BATCH_SIZE_EVAL = 8

GRADIENT_ACCUMULATION_STEPS = max(1, EFFECTIVE_BATCH_SIZE // BATCH_SIZE_TRAIN)
OPTIM = "adamw_torch_fused"

# ----- Label list (must match dataset exactly) -----
ALL_LABELS = [
    "Klima / Energie", "Zuwanderung", "Renten", "Soziales Gefälle",
    "AfD/Rechte", "Arbeitslosigkeit", "Wirtschaftslage", "Politikverdruss",
    "Gesundheitswesen, Pflege", "Kosten/Löhne/Preise",
    "Ukraine/Krieg/Russland", "Bundeswehr/Verteidigung", "Andere",
]

# ----- Model info (for report) -----
MODEL_INFO = {
    "huggingface_id": MODEL_ID,
    "language": "Multilingual (incl. German)",
    "max_tokens": MAX_LENGTH,
    "parameters": "210M",
    "notes": "EuroBERT-210m baseline fine-tuned with 3-fold CV + final retrain.",
}

print(f"GPU: {torch.cuda.get_device_name(0)} ({_gpu_mem:.1f} GB, CC {_gpu_cap[0]}.{_gpu_cap[1]})")
print(f"  BF16={USE_BF16}, FP16={USE_FP16}, Eval Batch={BATCH_SIZE_EVAL}")
print(f"\nBaseline hyperparameters:")
print(f"  LR:              {LEARNING_RATE:.2e}")
print(f"  Scheduler:       {LR_SCHEDULER_TYPE}")
print(f"  Epochs:          {NUM_EPOCHS}")
print(f"  Batch (train):   {BATCH_SIZE_TRAIN}")
print(f"  Grad Accum:      {GRADIENT_ACCUMULATION_STEPS}")
print(f"  Effective BS:    {EFFECTIVE_BATCH_SIZE}")
print(f"  Warmup Ratio:    {WARMUP_RATIO}")
print(f"  Weight Decay:    {WEIGHT_DECAY}")
print(f"  max_grad_norm:   {MAX_GRAD_NORM}")
print(f"  Early Stopping:  patience={EARLY_STOPPING_PATIENCE}")
print(f"  CV Folds:        {N_FOLDS}")
print(f"  Labels:          {len(ALL_LABELS)} classes")

In [ ]:
# ===== LOAD DATA & CREATE SPLITS =====
# Identical test split logic as notebooks 02 and 04.
# Remaining data is kept as cv_pool for 3-fold CV,
# and also split into train/val (80/20) for the final retrain phase.

import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

np.random.seed(RANDOM_SEED)

ds = load_dataset(pu.DATASET_ID)
train_hf = ds["train"].to_pandas()
test_hf = ds["test"].to_pandas()
all_labelled = pd.concat([train_hf, test_hf], ignore_index=True)

print(f"Total pool of labelled articles: {len(all_labelled)}")
print(f"Classes: {all_labelled['label'].nunique()}\n")

# --- Step 1: Fixed stratified test split (identical to 02/04) ---
test_indices = []
rest_indices = []

for label in ALL_LABELS:
    label_mask = all_labelled["label"] == label
    label_indices = all_labelled[label_mask].index.tolist()
    n_total = len(label_indices)
    if n_total < 60:
        n_test = n_total // 2
        print(f"  {label}: only {n_total} articles -> {n_test} for test")
    else:
        n_test = TEST_PER_CLASS
    np.random.shuffle(label_indices)
    test_indices.extend(label_indices[:n_test])
    rest_indices.extend(label_indices[n_test:])

test_df = all_labelled.loc[test_indices].reset_index(drop=True)
cv_pool_df = all_labelled.loc[rest_indices].reset_index(drop=True)

print(f"\nTest split:  {len(test_df)} articles (frozen)")
print(f"CV pool:     {len(cv_pool_df)} articles (for 3-fold CV + final retrain)")

# --- Step 2: Train/Val split for final retrain (identical to 04) ---
class_counts = cv_pool_df["label"].value_counts()
small_classes = class_counts[class_counts < 2].index.tolist()

if small_classes:
    print(f"\nClasses with <2 articles (fully assigned to Train): {small_classes}")
    small_mask = cv_pool_df["label"].isin(small_classes)
    train_small = cv_pool_df[small_mask]
    rest_for_split = cv_pool_df[~small_mask]
else:
    train_small = pd.DataFrame(columns=cv_pool_df.columns)
    rest_for_split = cv_pool_df

train_main, val_df = train_test_split(
    rest_for_split, test_size=VAL_FRACTION,
    stratify=rest_for_split["label"], random_state=RANDOM_SEED,
)
train_df = pd.concat([train_main, train_small], ignore_index=True)
val_df = val_df.reset_index(drop=True)

print(f"\n{'='*50}")
print(f"  CV Pool:    {len(cv_pool_df):>5} articles (for 3-fold CV)")
print(f"  Train:      {len(train_df):>5} articles (for final retrain)")
print(f"  Validation: {len(val_df):>5} articles (for final retrain)")
print(f"  Test:       {len(test_df):>5} articles (frozen)")
print(f"  Total:      {len(train_df) + len(val_df) + len(test_df):>5} articles")
print(f"{'='*50}")

# Class distribution overview
split_overview = pd.DataFrame({
    "CV Pool": cv_pool_df["label"].value_counts(),
    "Train": train_df["label"].value_counts(),
    "Val": val_df["label"].value_counts(),
    "Test": test_df["label"].value_counts(),
}).fillna(0).astype(int)
split_overview["Total"] = split_overview[["Train", "Val", "Test"]].sum(axis=1)
split_overview.loc["TOTAL"] = split_overview.sum()
print("\nClass distribution:")
print(split_overview.to_string())

# Split config for report
split_config = {
    "dataset_id": pu.DATASET_ID,
    "split_mode": "custom_finetune_cv",
    "test_per_class": TEST_PER_CLASS,
    "val_fraction": VAL_FRACTION,
    "n_folds": N_FOLDS,
    "random_seed": RANDOM_SEED,
    "cv_pool_size": len(cv_pool_df),
    "train_size": len(train_df),
    "eval_size": len(val_df),
    "test_size": len(test_df),
    "raw_size": 0,
}

In [ ]:
# ===== LABEL ENCODING =====
label2id = {label: idx for idx, label in enumerate(ALL_LABELS)}
id2label = {idx: label for idx, label in enumerate(ALL_LABELS)}

# Add numeric label_id to all DataFrames
cv_pool_df["label_id"] = cv_pool_df["label"].map(label2id)
train_df["label_id"] = train_df["label"].map(label2id)
val_df["label_id"] = val_df["label"].map(label2id)
test_df["label_id"] = test_df["label"].map(label2id)

# Sanity check: no unknown labels
for name, _df in [("CV Pool", cv_pool_df), ("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    assert _df["label_id"].isna().sum() == 0, f"Unknown labels in {name}!"

print("Label mapping:")
for label, idx in label2id.items():
    print(f"  {idx:>2}: {label}")
print(f"\nNumber of classes: {len(ALL_LABELS)}")

In [ ]:
# ===== TOKENIZATION =====
# Tokenize test set (shared across CV and final retrain).
# Also prepare train/val datasets for final retrain phase.
# CV fold tokenization happens inside the fold loop.

from transformers import AutoTokenizer
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

def tokenize_fn(examples):
    return tokenizer(examples["text"], max_length=MAX_LENGTH, truncation=True)

# Test dataset (frozen, used once after final retrain)
test_dataset = Dataset.from_pandas(test_df[["text", "label_id"]].rename(columns={"label_id": "labels"}))
print("Tokenizing test data...")
test_dataset = test_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# Train/Val datasets for final retrain phase
train_dataset = Dataset.from_pandas(train_df[["text", "label_id"]].rename(columns={"label_id": "labels"}))
val_dataset = Dataset.from_pandas(val_df[["text", "label_id"]].rename(columns={"label_id": "labels"}))
print("Tokenizing train data...")
train_dataset = train_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
print("Tokenizing validation data...")
val_dataset = val_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print(f"\nDataset sizes:")
print(f"  Train:      {len(train_dataset)} (for final retrain)")
print(f"  Validation: {len(val_dataset)} (for final retrain)")
print(f"  Test:       {len(test_dataset)} (frozen)")

In [ ]:
# ===== ROPE FIX & MODEL FACTORY =====
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, AutoConfig
from transformers.modeling_rope_utils import ROPE_INIT_FUNCTIONS

# EuroBERT RoPE fix (required for all EuroBERT notebooks)
def _default_rope_init(config, device=None, **kwargs):
    base = getattr(config, "rope_theta", 10000.0)
    partial_rotary_factor = getattr(config, "partial_rotary_factor", 1.0)
    head_dim = getattr(config, "head_dim", config.hidden_size // config.num_attention_heads)
    dim = int(head_dim * partial_rotary_factor)
    inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.int64).float().to(device) / dim))
    return inv_freq, 1.0

ROPE_INIT_FUNCTIONS["default"] = _default_rope_init


def create_model(seed=42):
    """Create a fresh model with deterministic initialization.

    NaN prevention:
    1. Deterministic seeding for reproducible classifier head init
    2. Re-init with smaller std (0.002) to prevent logit overflow in BF16
    """
    torch.manual_seed(seed)

    config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
    config.num_labels = len(ALL_LABELS)
    config.id2label = id2label
    config.label2id = label2id

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, config=config,
        ignore_mismatched_sizes=True, trust_remote_code=True,
    )

    # Stabilize classifier head: smaller init prevents logit overflow in BF16
    for name, module in model.named_modules():
        if name in ("dense", "classifier") and isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.002)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    return model


# Count parameters (quick check)
_tmp = create_model()
total_params = sum(p.numel() for p in _tmp.parameters())
trainable_params = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
del _tmp

print("RoPE fix applied.")
print("create_model() defined.")
print(f"Total parameters:     {total_params:>12,}")
print(f"Trainable parameters: {trainable_params:>12,}")
print(f"GPU: {pu.get_gpu_info()['gpu_name']} ({pu.get_gpu_info()['gpu_vram_gb']} GB)")

In [ ]:
# ===== METRICS FUNCTION =====
# Use the shared factory from hpt_utils to ensure metric consistency across all pipeline notebooks
compute_metrics = hu.make_compute_metrics(ALL_LABELS, id2label)

print("compute_metrics created via hu.make_compute_metrics().")

In [ ]:
# ===== 3-FOLD STRATIFIED CROSS-VALIDATION =====
# Identical CV strategy as notebook 02: StratifiedKFold on the CV pool,
# fresh model per fold, NaN detection via HPTMetricsCallback.

import gc
import math
import shutil
from sklearn.model_selection import StratifiedKFold
from transformers import (
    TrainingArguments, Trainer, EarlyStoppingCallback, DataCollatorWithPadding,
)

# Pre-compute fold indices (same folds every run for reproducibility)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
fold_indices = list(skf.split(cv_pool_df, cv_pool_df["label_id"]))

print(f"Stratified {N_FOLDS}-fold CV on {len(cv_pool_df)} articles:")
for i, (train_idx, val_idx) in enumerate(fold_indices):
    val_labels = cv_pool_df.iloc[val_idx]["label"].value_counts()
    print(f"  Fold {i+1}: Train={len(train_idx)}, Val={len(val_idx)}, "
          f"smallest val class: {val_labels.idxmin()} ({val_labels.min()} samples)")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# --- CV fold loop ---
fold_results = []
cv_timer = pu.ExperimentTimer()

with cv_timer:
    for fold_idx, (train_idx, val_idx) in enumerate(fold_indices):
        print(f"\n{'='*60}")
        print(f"  FOLD {fold_idx + 1}/{N_FOLDS}")
        print(f"{'='*60}")

        # Clear VRAM between folds
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        # Extract fold data from CV pool
        fold_train_df = cv_pool_df.iloc[train_idx]
        fold_val_df = cv_pool_df.iloc[val_idx]

        # Tokenize fold data
        fold_train_ds = Dataset.from_pandas(
            fold_train_df[["text", "label_id"]].rename(columns={"label_id": "labels"})
        )
        fold_val_ds = Dataset.from_pandas(
            fold_val_df[["text", "label_id"]].rename(columns={"label_id": "labels"})
        )
        fold_train_ds = fold_train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
        fold_val_ds = fold_val_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
        fold_train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
        fold_val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

        # Fresh model for this fold (matching seed scheme from notebook 02)
        fold_seed = RANDOM_SEED + fold_idx
        model = create_model(seed=fold_seed)
        model = model.to(torch.device("cuda"))

        # Compute warmup steps
        steps_per_epoch = math.ceil(
            len(fold_train_ds) / (BATCH_SIZE_TRAIN * GRADIENT_ACCUMULATION_STEPS)
        )
        total_steps = steps_per_epoch * NUM_EPOCHS
        warmup_steps = int(total_steps * WARMUP_RATIO)

        # NaN detection callback
        nan_callback = hu.HPTMetricsCallback(
            trial_number=0,
            fold_idx=fold_idx,
            all_labels=ALL_LABELS,
            id2label=id2label,
        )

        fold_output_dir = f"/content/baseline_cv_fold_{fold_idx}"

        training_args = TrainingArguments(
            output_dir=fold_output_dir,
            num_train_epochs=NUM_EPOCHS,
            learning_rate=LEARNING_RATE,
            lr_scheduler_type=LR_SCHEDULER_TYPE,
            per_device_train_batch_size=BATCH_SIZE_TRAIN,
            per_device_eval_batch_size=BATCH_SIZE_EVAL,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
            warmup_steps=warmup_steps,
            weight_decay=WEIGHT_DECAY,
            label_smoothing_factor=LABEL_SMOOTHING,
            fp16=USE_FP16,
            bf16=USE_BF16,
            gradient_checkpointing=False,
            optim=OPTIM,
            max_grad_norm=MAX_GRAD_NORM,
            group_by_length=True,
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=1,
            load_best_model_at_end=True,
            metric_for_best_model="f1_macro",
            greater_is_better=True,
            logging_strategy="steps",
            logging_steps=LOGGING_STEPS,
            report_to="none",
            seed=fold_seed,
            dataloader_num_workers=4,
            dataloader_pin_memory=True,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=fold_train_ds,
            eval_dataset=fold_val_ds,
            data_collator=data_collator,
            compute_metrics=compute_metrics,
            callbacks=[
                EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE),
                nan_callback,
            ],
        )

        trainer.train()

        # Check for NaN or collapse
        if nan_callback.nan_detected or nan_callback.lr_zero_detected:
            reason = "NaN" if nan_callback.nan_detected else "LR=0"
            print(f"  [{reason} DETECTED] Fold {fold_idx + 1} failed!")
            fold_results.append({"fold": fold_idx + 1, "status": "failed", "reason": reason})
        else:
            eval_result = trainer.evaluate()
            fold_f1 = eval_result.get("eval_f1_macro", 0)
            fold_acc = eval_result.get("eval_accuracy", 0)
            fold_prec = eval_result.get("eval_precision_macro", 0)
            fold_rec = eval_result.get("eval_recall_macro", 0)

            if math.isnan(eval_result.get("eval_loss", 0)) or fold_f1 < 0.05:
                print(f"  [COLLAPSE] Fold {fold_idx + 1}: F1={fold_f1:.4f}")
                fold_results.append({"fold": fold_idx + 1, "status": "failed", "reason": "collapse"})
            else:
                fold_results.append({
                    "fold": fold_idx + 1,
                    "status": "success",
                    "f1_macro": fold_f1,
                    "accuracy": fold_acc,
                    "precision_macro": fold_prec,
                    "recall_macro": fold_rec,
                })
                print(f"  Fold {fold_idx + 1}: F1 Macro = {fold_f1:.4f}, "
                      f"Acc = {fold_acc:.4f}, Prec = {fold_prec:.4f}, Rec = {fold_rec:.4f}")

        # Cleanup fold artifacts
        del trainer, model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        if os.path.exists(fold_output_dir):
            shutil.rmtree(fold_output_dir, ignore_errors=True)

print(f"\nCV complete. Duration: {cv_timer.duration_formatted}")

In [ ]:
# ===== CV RESULTS SUMMARY =====
import matplotlib.pyplot as plt

successful_folds = [r for r in fold_results if r["status"] == "success"]
failed_folds = [r for r in fold_results if r["status"] == "failed"]

if failed_folds:
    print(f"WARNING: {len(failed_folds)} fold(s) failed!")
    for f in failed_folds:
        print(f"  Fold {f['fold']}: {f['reason']}")

if not successful_folds:
    raise RuntimeError("All CV folds failed! Cannot proceed.")

# Aggregate metrics across successful folds
cv_metrics = {
    "f1_macro":        [r["f1_macro"] for r in successful_folds],
    "accuracy":        [r["accuracy"] for r in successful_folds],
    "precision_macro": [r["precision_macro"] for r in successful_folds],
    "recall_macro":    [r["recall_macro"] for r in successful_folds],
}

print(f"\n{'='*60}")
print(f"  BASELINE 3-FOLD CV RESULTS ({len(successful_folds)}/{N_FOLDS} folds)")
print(f"{'='*60}")
print(f"{'Metric':<20} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8}")
print("-" * 55)
for metric_name, values in cv_metrics.items():
    print(f"{metric_name:<20} {np.mean(values):>8.4f} {np.std(values):>8.4f} "
          f"{np.min(values):>8.4f} {np.max(values):>8.4f}")
print(f"{'='*60}")

# Per-fold table
print(f"\nPer-fold breakdown:")
print(f"{'Fold':>6} {'F1 Macro':>10} {'Accuracy':>10} {'Precision':>10} {'Recall':>10}")
print("-" * 50)
for r in successful_folds:
    print(f"{r['fold']:>6} {r['f1_macro']:>10.4f} {r['accuracy']:>10.4f} "
          f"{r['precision_macro']:>10.4f} {r['recall_macro']:>10.4f}")

# Store CV summary for report
cv_summary = {
    "n_folds": N_FOLDS,
    "folds_completed": len(successful_folds),
    "mean_f1_macro": round(float(np.mean(cv_metrics["f1_macro"])), 4),
    "std_f1_macro": round(float(np.std(cv_metrics["f1_macro"])), 4),
    "mean_accuracy": round(float(np.mean(cv_metrics["accuracy"])), 4),
    "std_accuracy": round(float(np.std(cv_metrics["accuracy"])), 4),
    "mean_precision_macro": round(float(np.mean(cv_metrics["precision_macro"])), 4),
    "std_precision_macro": round(float(np.std(cv_metrics["precision_macro"])), 4),
    "mean_recall_macro": round(float(np.mean(cv_metrics["recall_macro"])), 4),
    "std_recall_macro": round(float(np.std(cv_metrics["recall_macro"])), 4),
    "per_fold_f1": [round(f, 4) for f in cv_metrics["f1_macro"]],
}

# Bar chart: fold comparison
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(successful_folds))
width = 0.2

ax.bar(x - 1.5*width, [r["f1_macro"] for r in successful_folds], width, label="F1 Macro", color="#4CAF50")
ax.bar(x - 0.5*width, [r["accuracy"] for r in successful_folds], width, label="Accuracy", color="#2196F3")
ax.bar(x + 0.5*width, [r["precision_macro"] for r in successful_folds], width, label="Precision", color="#FF9800")
ax.bar(x + 1.5*width, [r["recall_macro"] for r in successful_folds], width, label="Recall", color="#9C27B0")

ax.set_xticks(x)
ax.set_xticklabels([f"Fold {r['fold']}" for r in successful_folds])
ax.set_ylabel("Score")
ax.set_title("Baseline 3-Fold CV: Per-Fold Metrics", fontsize=13, fontweight="bold")
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.3)

# Add mean line
mean_f1 = np.mean(cv_metrics["f1_macro"])
ax.axhline(y=mean_f1, color="red", linestyle="--", alpha=0.7, label=f"Mean F1={mean_f1:.4f}")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ===== FINAL RETRAIN WITH NaN AUTO-RETRY =====
# Train on full train set (80% of non-test data), validate on val set (20%).
# Matching the NaN retry pattern from notebook 04.

OUTPUT_DIR = "/content/baseline_final_output"


def run_training(attempt_seed):
    """Single training attempt. Returns (trainer, success)."""
    print(f"\n{'='*60}")
    print(f"  Final retrain attempt with seed {attempt_seed}")
    print(f"{'='*60}")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    model = create_model(seed=attempt_seed)
    model = model.to(torch.device("cuda"))

    steps_per_epoch = math.ceil(
        len(train_dataset) / (BATCH_SIZE_TRAIN * GRADIENT_ACCUMULATION_STEPS)
    )
    total_steps = steps_per_epoch * NUM_EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)

    nan_callback = hu.HPTMetricsCallback(
        trial_number=0,
        fold_idx=attempt_seed,
        all_labels=ALL_LABELS,
        id2label=id2label,
    )

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type=LR_SCHEDULER_TYPE,
        per_device_train_batch_size=BATCH_SIZE_TRAIN,
        per_device_eval_batch_size=BATCH_SIZE_EVAL,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        warmup_steps=warmup_steps,
        weight_decay=WEIGHT_DECAY,
        label_smoothing_factor=LABEL_SMOOTHING,
        fp16=USE_FP16,
        bf16=USE_BF16,
        gradient_checkpointing=False,
        optim=OPTIM,
        max_grad_norm=MAX_GRAD_NORM,
        group_by_length=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=LOGGING_STEPS,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=3,
        report_to="none",
        seed=attempt_seed,
        dataloader_num_workers=4,
        dataloader_pin_memory=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[
            EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE),
            nan_callback,
        ],
    )

    print(f"  Epochs: {NUM_EPOCHS}, LR: {LEARNING_RATE:.2e}, Warmup: {warmup_steps} steps")
    print(f"  Batch: {BATCH_SIZE_TRAIN} x {GRADIENT_ACCUMULATION_STEPS} = {EFFECTIVE_BATCH_SIZE}")

    trainer.train()

    # NaN / collapse check
    if nan_callback.nan_detected or nan_callback.lr_zero_detected:
        reason = "NaN" if nan_callback.nan_detected else "LR=0"
        print(f"  [{reason} DETECTED] Training aborted.")
        return trainer, False

    eval_result = trainer.evaluate()
    eval_f1 = eval_result.get("eval_f1_macro", 0)
    eval_loss = eval_result.get("eval_loss", 0)

    if (eval_loss is not None and (math.isnan(eval_loss) or math.isinf(eval_loss))) \
       or eval_f1 < 0.05:
        print(f"  [COLLAPSE] Eval F1={eval_f1:.4f}, Loss={eval_loss} -> Training failed.")
        return trainer, False

    print(f"  Training successful! Eval F1 Macro: {eval_f1:.4f}")
    return trainer, True


# === NaN retry loop ===
timer = pu.ExperimentTimer()
trainer = None
training_success = False

with timer:
    for attempt in range(MAX_NAN_RETRIES):
        attempt_seed = RANDOM_SEED + attempt * 1000
        print(f"\n>>> Attempt {attempt + 1}/{MAX_NAN_RETRIES} (seed={attempt_seed})")

        if trainer is not None:
            del trainer
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

        if os.path.exists(OUTPUT_DIR):
            shutil.rmtree(OUTPUT_DIR, ignore_errors=True)

        trainer, training_success = run_training(attempt_seed)

        if training_success:
            print(f"\n>>> Training successful after {attempt + 1} attempt(s)!")
            break
        else:
            print(f"\n>>> Attempt {attempt + 1} failed. ", end="")
            if attempt < MAX_NAN_RETRIES - 1:
                print("Starting next attempt with new seed...")
            else:
                print("All attempts exhausted!")

if not training_success:
    raise RuntimeError(
        f"Training failed after {MAX_NAN_RETRIES} attempts! "
        f"Possible causes: NaN, LR=0, model collapse."
    )

print(f"\nFinal retrain complete. Duration: {timer.duration_formatted}")
print(f"Best model: {trainer.state.best_model_checkpoint}")
print(f"Best metric (F1 Macro): {trainer.state.best_metric:.4f}")

In [ ]:
# ===== FINAL RETRAIN: TRAINING HISTORY =====
log_history = trainer.state.log_history

train_steps = [e["step"] for e in log_history if "loss" in e]
train_losses = [e["loss"] for e in log_history if "loss" in e]

eval_logs = [e for e in log_history if "eval_loss" in e]
eval_epochs = [e["epoch"] for e in eval_logs]
eval_losses = [e["eval_loss"] for e in eval_logs]
eval_f1 = [e.get("eval_f1_macro", 0) for e in eval_logs]
eval_acc = [e.get("eval_accuracy", 0) for e in eval_logs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

ax1.plot(train_steps, train_losses, alpha=0.4, label="Train Loss", color="steelblue")
eval_steps = [e.get("step", 0) for e in eval_logs]
ax1.plot(eval_steps, eval_losses, "o-", label="Eval Loss", color="orangered", linewidth=2)
ax1.set_xlabel("Steps")
ax1.set_ylabel("Loss")
ax1.set_title("Training vs. Eval Loss (Final Retrain)")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(eval_epochs, eval_f1, "o-", label="F1 Macro", linewidth=2)
ax2.plot(eval_epochs, eval_acc, "s-", label="Accuracy", linewidth=2)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Score")
ax2.set_title("Eval Metrics per Epoch (Final Retrain)")
ax2.legend(loc="lower right")
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1)

plt.suptitle("EuroBERT-210m Baseline: Final Retrain History", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\nEval metrics per epoch:")
print(f"{'Epoch':>6} {'Loss':>8} {'F1 Macro':>10} {'Accuracy':>10}")
print("-" * 40)
for log in eval_logs:
    print(f"{log['epoch']:>6.0f} {log['eval_loss']:>8.4f} "
          f"{log.get('eval_f1_macro', 0):>10.4f} {log.get('eval_accuracy', 0):>10.4f}")

In [ ]:
# ===== EVALUATION ON TEST SET =====
print("Evaluating on test set with best model...")

test_preds = trainer.predict(test_dataset)
pred_ids = np.argmax(test_preds.predictions, axis=-1)
pred_labels = [id2label[i] for i in pred_ids]
true_labels = [id2label[i] for i in test_preds.label_ids]

metrics = pu.evaluate(
    true_labels, pred_labels,
    labels=ALL_LABELS, experiment_name="test",
)
pu.print_metrics(metrics, "Baseline EuroBERT-210m -- Test Split")

In [ ]:
# ===== CONFUSION MATRIX =====
pu.plot_confusion_matrix(metrics, title="Baseline EuroBERT-210m (Test)")

In [ ]:
# ===== PER-CLASS METRICS BARPLOT =====
pc_df = metrics["per_class_df"].copy().sort_values("F1", ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
y_pos = np.arange(len(pc_df))
bar_h = 0.25

ax.barh(y_pos - bar_h, pc_df["Precision"], bar_h, label="Precision", color="#2196F3", alpha=0.85)
ax.barh(y_pos, pc_df["Recall"], bar_h, label="Recall", color="#FF9800", alpha=0.85)
ax.barh(y_pos + bar_h, pc_df["F1"], bar_h, label="F1", color="#4CAF50", alpha=0.85)

ax.set_yticks(y_pos)
ax.set_yticklabels(pc_df["Label"])
ax.set_xlabel("Score")
ax.set_title("Per-Class Metrics: Baseline EuroBERT-210m (Test)", fontsize=13, fontweight="bold")
ax.legend(loc="lower right")
ax.set_xlim(0, 1.05)
ax.grid(axis="x", alpha=0.3)

for i, (_, row) in enumerate(pc_df.iterrows()):
    ax.text(row["F1"] + 0.01, y_pos[i] + bar_h, f"{row['F1']:.2f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ===== GENERATE REPORT =====
training_params = {
    "source": "baseline (default hyperparameters)",
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "lr_scheduler_type": LR_SCHEDULER_TYPE,
    "batch_size_train": BATCH_SIZE_TRAIN,
    "batch_size_eval": BATCH_SIZE_EVAL,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "effective_batch_size": EFFECTIVE_BATCH_SIZE,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "label_smoothing_factor": LABEL_SMOOTHING,
    "max_length": MAX_LENGTH,
    "max_grad_norm": MAX_GRAD_NORM,
    "bf16": USE_BF16,
    "fp16": USE_FP16,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "gradient_checkpointing": False,
    "optimizer": OPTIM,
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_metric": round(trainer.state.best_metric, 4) if trainer.state.best_metric else None,
    # CV results
    "cv_n_folds": cv_summary["n_folds"],
    "cv_folds_completed": cv_summary["folds_completed"],
    "cv_mean_f1_macro": cv_summary["mean_f1_macro"],
    "cv_std_f1_macro": cv_summary["std_f1_macro"],
    "cv_mean_accuracy": cv_summary["mean_accuracy"],
    "cv_std_accuracy": cv_summary["std_accuracy"],
    "cv_per_fold_f1": cv_summary["per_fold_f1"],
}

config_dict = trainer.model.config.to_dict()
model_config = {}
for field in ["architectures", "model_type", "hidden_size", "num_hidden_layers",
              "num_attention_heads", "vocab_size", "max_position_embeddings"]:
    if field in config_dict:
        val = config_dict[field]
        if field == "architectures" and isinstance(val, list):
            val = val[0] if len(val) == 1 else ", ".join(val)
        model_config[field] = val

report_path = pu.generate_report(
    model_name=f"{MODEL_SHORT_NAME}_baseline",
    model_type=MODEL_TYPE,
    metrics=metrics,
    timer=timer,
    model_info=MODEL_INFO,
    candidate_labels=ALL_LABELS,
    split_config=split_config,
    label_mapping={l: l for l in ALL_LABELS},
    model_config=model_config,
    training_params=training_params,
    experiment_notes=(
        f"BASELINE: EuroBERT-210m fine-tuned with default hyperparameters. "
        f"3-fold stratified CV: F1 Macro = {cv_summary['mean_f1_macro']:.4f} "
        f"+/- {cv_summary['std_f1_macro']:.4f}. "
        f"Final retrain on {len(train_df)} articles, "
        f"validated on {len(val_df)}, tested on {len(test_df)}. "
        f"Custom split: {TEST_PER_CLASS} test/class, "
        f"remainder {int((1-VAL_FRACTION)*100)}/{int(VAL_FRACTION*100)} train/val."
    ),
)
print(f"Report saved: {report_path}")

In [ ]:
# ===== UPLOAD MODEL TO HUGGINGFACE =====
url = pu.upload_model_to_hub(
    model=trainer.model,
    tokenizer=tokenizer,
    repo_name=REPO_NAME,
    private=True,
    training_params=training_params,
)
print(f"Model uploaded: {url}")

In [ ]:
# ===== SUMMARY & CLEANUP =====
print("=" * 70)
print(f"  BASELINE FINE-TUNING COMPLETE")
print("=" * 70)
print(f"  Model:           {REPO_NAME}")
print(f"  Base:            {MODEL_ID}")
print(f"")
print(f"  --- 3-Fold CV Results ---")
print(f"  CV F1 Macro:     {cv_summary['mean_f1_macro']:.4f} +/- {cv_summary['std_f1_macro']:.4f}")
print(f"  CV Accuracy:     {cv_summary['mean_accuracy']:.4f} +/- {cv_summary['std_accuracy']:.4f}")
print(f"  Per-fold F1:     {cv_summary['per_fold_f1']}")
print(f"  CV Duration:     {cv_timer.duration_formatted}")
print(f"")
print(f"  --- Final Model (Test Set) ---")
print(f"  Train:           {len(train_df)} articles")
print(f"  Validation:      {len(val_df)} articles")
print(f"  Test:            {len(test_df)} articles")
print(f"  Epochs:          {NUM_EPOCHS} (best: {trainer.state.best_model_checkpoint})")
print(f"  F1 Macro:        {metrics['f1_macro']:.4f}")
print(f"  F1 Weighted:     {metrics['f1_weighted']:.4f}")
print(f"  Accuracy:        {metrics['accuracy']:.4f}")
print(f"  Precision Macro: {metrics['precision_macro']:.4f}")
print(f"  Recall Macro:    {metrics['recall_macro']:.4f}")
print(f"  Retrain Duration:{timer.duration_formatted}")
print(f"")
print(f"  Report:          {report_path}")
print(f"  HuggingFace:     {url}")
print("=" * 70)

# Cleanup
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
    print("\nCheckpoint files deleted.")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    free_mem = torch.cuda.mem_get_info()[0] / 1e9
    print(f"GPU VRAM free: {free_mem:.1f} GB")

print("\nDone. Runtime can be terminated.")